# Task 1: Language model inference

The goal if this first task is to familiarize yourself with the huggingface transformers and dataset libraries. You will learn how to load and tokenize a dataset, how to load a pre-trained language model, and finally, how to run a model in inference mode.

Your task is to complete the missing code blocks below.

In [1]:
# import dependencies
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from datasets import load_dataset, load_dataset_builder, get_dataset_split_names, get_dataset_config_names
from transformers import XGLMTokenizer, XGLMTokenizerFast, XGLMForCausalLM, AutoModelForCausalLM, AutoTokenizer, GenerationConfig

## Explore dataset

In [2]:
DATA_SET_NAME = "facebook/flores" # specify dataset name
MODEL_NAME = "facebook/xglm-564M" # specify model name
# MODEL_NAME = "gpt2" # specify model name

In [3]:
# Explore a dataset

# covered language codes can be found here: https://github.com/openlanguagedata/flores?tab=readme-ov-file#language-coverage

ds_builder = load_dataset_builder("facebook/flores", "deu_Latn")
print(ds_builder.info.description) # print the dataset description

/home/user/anaconda3/envs/nnti-project/lib/python3.12/site-packages/datasets/load.py:1461: FutureWarning: The repository for facebook/flores contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/facebook/flores
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


The creation of FLORES-200 doubles the existing language coverage of FLORES-101. 
Given the nature of the new languages, which have less standardization and require 
more specialized professional translations, the verification process became more complex. 
This required modifications to the translation workflow. FLORES-200 has several languages 
which were not translated from English. Specifically, several languages were translated 
from Spanish, French, Russian and Modern Standard Arabic. Moreover, FLORES-200 also 
includes two script alternatives for four languages. FLORES-200 consists of translations 
from 842 distinct web articles, totaling 3001 sentences. These sentences are divided 
into three splits: dev, devtest, and test (hidden). On average, sentences are approximately 
21 words long.



In [4]:
# print the features (columns) of the dataset
# TODO: your code goes here
print(ds_builder.info.features)

{'id': Value(dtype='int32', id=None), 'URL': Value(dtype='string', id=None), 'domain': Value(dtype='string', id=None), 'topic': Value(dtype='string', id=None), 'has_image': Value(dtype='int32', id=None), 'has_hyperlink': Value(dtype='int32', id=None), 'sentence': Value(dtype='string', id=None)}


In [5]:
# get the available splits
# TODO: your code goes here
#print(ds_builder.keys())


## Load data, tokenize, and batchify

In [6]:
# specify languages
LANGUAGES = [
    "eng_Latn",
    "spa_Latn",
    "ita_Latn",
    "deu_Latn",
    "arb_Arab",
    "tel_Telu",
    "tam_Taml",
    "quy_Latn"
]

In [7]:
# load flores data for each language
# TODO: your code goes here
dataset = []
for language in LANGUAGES:
    dataset.append(load_dataset("facebook/flores", language, trust_remote_code=True))

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

...

Generating devtest split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [8]:
# let's look at the English subset
# TODO: your code goes here
eng = dataset[0]
print(eng)

DatasetDict({
    dev: Dataset({
        features: ['id', 'URL', 'domain', 'topic', 'has_image', 'has_hyperlink', 'sentence'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'URL', 'domain', 'topic', 'has_image', 'has_hyperlink', 'sentence'],
        num_rows: 1012
    })
})


In [9]:
# let's look at an individal sample from the dataset
# TODO: your code goes here

# check the first sample from the dev set
print(eng["dev"][0])

# check the first sample from the devtest set
print(eng["devtest"][0])

{'id': 1, 'URL': 'https://en.wikinews.org/wiki/Scientists_say_new_medical_diagnostic_chip_can_sort_cells_anywhere_with_an_inkjet', 'domain': 'wikinews', 'topic': 'health', 'has_image': 0, 'has_hyperlink': 0, 'sentence': 'On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'}
{'id': 1, 'URL': 'https://en.wikinews.org/wiki/Toronto_team-led_research_on_Type_1_Diabetes_%27groundbreaking%27', 'domain': 'wikinews', 'topic': 'disease, research, canada', 'has_image': 0, 'has_hyperlink': 0, 'sentence': '"We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.'}


In [10]:
# tokenize the data
# The model is either facebook/xglm-564M or gpt2.

# load a pre-trained tokenizer from the huggingface hub

print("Model name: ", MODEL_NAME)
# The from_pretrained() method takes care of returning the correct tokenizer class instance based on the model_type property
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# since we are using a decoder only architecture, we need to perform the padding
# on the left side so that the model does not get confused while guessing the next token.
tokenizer.padding_side = 'left'

print(tokenizer)

# gpt2 does not have a padding token, so we have to add it manually
if MODEL_NAME == "gpt2":
    tokenizer.add_special_tokens({'pad_token': tokenizer.unk_token})

#tokenizer.config.output_attentions

# specify the tokenization function
def tokenization(example):
    # fill in here
    tokenized = tokenizer(example)
    return tokenized

# TODO: your code goes here
first_sentence = eng["devtest"][0]["sentence"]
print(first_sentence)
tokenized_sentence = tokenization(first_sentence)

Model name:  facebook/xglm-564M
XGLMTokenizerFast(name_or_path='facebook/xglm-564M', vocab_size=256008, model_max_length=2048, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'additional_special_tokens': ['<madeupword0>', '<madeupword1>', '<madeupword2>', '<madeupword3>', '<madeupword4>', '<madeupword5>', '<madeupword6>']}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	256001: AddedToken("<madeupword0>", rstrip=Fal

In [11]:
# let's take a look at a tokenized sample
# TODO: your code goes here

In [12]:
# construct a pytorch data loader for each dataset
BATCH_SIZE = 2 # for testing purposes, we start with a batch size of 2. You can change this later.

# TODO: your code goes here
# I assume that each language is making up a dataset.

# Create a dictionary of DataLoaders for each language and split
data_loaders = {
    lang: {
        "dev": DataLoader(lang_dataset["dev"], batch_size=BATCH_SIZE, shuffle=True),
        "devtest": DataLoader(lang_dataset["devtest"], batch_size=BATCH_SIZE, shuffle=True)
    } for lang, lang_dataset in zip(LANGUAGES, dataset)
}


## Load model

In [13]:
# load pre-trained model from the huggingface hub
print(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
# put the model into evaluation mode
# TODO: your code goes here
model.eval()

facebook/xglm-564M


XGLMForCausalLM(
  (model): XGLMModel(
    (embed_tokens): Embedding(256008, 1024, padding_idx=1)
    (embed_positions): XGLMSinusoidalPositionalEmbedding()
    (layers): ModuleList(
      (0-23): 24 x XGLMDecoderLayer(
        (self_attn): XGLMAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (activation_fn): GELUActivation()
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (final_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      )
    )
    (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine

In [ ]:
losses = {lang: [] for lang in LANGUAGES} # store per-batch losses for each language

# iterate over the dataset for each language and compute the cross-entropy loss per batch
# TODO: your code goes here

# check if the current model is capable of generating new tokens.
print(model.can_generate())

# Find the loss for every language individually.
for lang in LANGUAGES:
    # Find the loss based on dev set
    current_loader = data_loaders[lang]["dev"]

    # set the loss to zero initially
    loss = 0

    # iterate over the batches and find the loss per batch
    for i in range(len(current_loader)):
        # since batch size = 2, we can get 2 new instances from the dataset.
        sentences = next(iter(current_loader))["sentence"]

        # encode the sentence (without the last word)
        encoded_sentences = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")

        # get the input ids
        input_ids = encoded_sentences["input_ids"]

        # attention mask either holds a value of 0 or 1 depending on
        # if the token is a padded one or not.
        attention_mask = encoded_sentences["attention_mask"]

        # we need to put a label of -100 for the padded parts
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        # pass the input ids and labels to the model
        generated_output = model(input_ids=input_ids, labels=labels)

        # get the loss
        current_loss = generated_output.loss
        loss += current_loss

    losses[lang] = loss / len(current_loader)



True


## Visualize loss per language

In [ ]:
# create a figure
fig, axes = plt.subplots(figsize=(8, 5))

# create a bar plot for each langauge
# TODO: your code goes here

# format plot
axes.set_xlabel("language") # x-axis label
axes.set_xticks(range(len(LANGUAGES))) # x-axis ticks
axes.set_xticklabels(losses.keys()) # x-axis tick labels
axes.set_ylabel("loss") # y-axis label
axes.set_ylim(0, 9) # range of y-axis
axes.set_title(MODEL_NAME); # title

## Comparing XGLM to GPT2

Your next task is to re-run the analysis above, but using `gpt2` as the pre-trained language model. For this exercise, focus on your native language, unless it's English or isn't covered by flores. In that case, pick another language that you can read well.

Compare the language modeling loss of XGLM and GPT2. What do you observe? Investigate the differences in tokenization for XGLM and GPT2. What do you observe? How can the good (or bad) performance of GPT2 be explained?

In [ ]:
# TODO: your code goes here